# Recursive CTE Practice — Sales by Product & Year

**Question:** Given a table `b_sql.b_practice.sales_re` with columns `product_id`, `period_start`, `period_end`, and `average_daily_sales`, calculate the **total sales per product per year**.

Each product has a date range (`period_start` to `period_end`) and an `average_daily_sales` value that applies to every day in that range. To aggregate by year, you need to expand each product's date range into individual dates, then group by product and year.

**Approach:**
1. Use a **recursive CTE** to generate all dates from `min(period_start)` to `max(period_end)`
2. Join the date sequence with the sales table on dates falling within each product's period
3. Group by `product_id` and `year` to sum `average_daily_sales`

An equivalent **PySpark** solution using `sequence()` and `explode()` is also provided.

In [0]:
%sql
-- Basic recursive CTE: generate numbers 1 through 6 using UNION ALL
with recursive cte as (
    select 1 as num
    union all 
    select num+1 as num from cte 
    where num<6
)
select * from cte

In [0]:
# PySpark equivalent: number sequence using sequence() and explode()
from pyspark.sql import functions

data = [
    (2, 5),
    (4, 7),
    (6, 9)
]

df = spark.createDataFrame(data, ["start_num", "end_num"])

df.show()
from pyspark.sql.functions import min, max, sequence, explode, lit

df3 = (
    df
    .select(
        min("start_num").alias("start_num"),
        max("end_num").alias("end_num")
    )
    .select(
        explode(
            sequence("start_num", "end_num", lit(1))
        ).alias("num")
    )
)

df3.show()



In [0]:


# Alternative: generate numbers using spark.range()
df = (
    spark.range(1, 5001)
    .select(col("id").alias("num"))
)

display(df)


In [0]:
%sql
-- Create the sales_re table and insert sample sales data
create table b_sql.b_practice.sales_re (
product_id int,
period_start date,
period_end date,
average_daily_sales int
);

insert into b_sql.b_practice.sales_re values(1,'2019-01-25','2019-02-28',100),(2,'2018-12-01','2020-01-01',10),(3,'2019-12-01','2020-01-31',1);


In [0]:
%sql
-- View the sales_re table records
select * from b_sql.b_practice.sales_re

In [0]:
%sql
-- Recursive CTE: generate dates, join with sales, aggregate by product and year
with recursive date_sales
max recursion level 5000
as(
    select min(period_start) as start_date, max(period_end) as end_date   from b_sql.b_practice.sales_re
    union all 
    select cast(dateadd(day,1,start_date) as date) as start_date,end_date from date_sales where start_date<end_date

)
-- select * from date_sales
,product_sales as (select s.product_id,year(d.start_date) as year,d.start_date,average_daily_sales as sales from b_sql.b_practice.sales_re s join date_sales d on d.start_date between s.period_start and s.period_end order by product_id)
select product_id,year,sum(sales) as sales from product_sales group by product_id,year order by product_id,year


In [0]:
# Import PySpark functions and types for DataFrame operations
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
# Read the sales_re table into a PySpark DataFrame
df_sale=_sales=spark.read.table("b_sql.b_practice.sales_re")
df_sale.display()

In [0]:
# PySpark: generate date sequence using sequence() and explode()
df_re=(df_sale.selectExpr("min(period_start) as period_start"
                     ,"max(period_end) as period_end" )
            .withColumn("date_info", sequence(
                to_date("period_start"),to_date("period_end"),expr("interval 1 day")
                )
            ).withColumn("date_seq", explode("date_info"))
)
df_re.display()

In [0]:

# PySpark: join sales with dates, group by product_id and year, sum sales
df_final = (
    df_sale.alias("sale")
    .join(
        df_re.alias("re"),
        col("re.date_seq").between(
            col("sale.period_start"),
            col("sale.period_end")
        ),
        "inner"
    ).select("sale.product_id","sale.average_daily_sales","re.date_seq",year("re.date_seq").alias("year"))
)
df_final=df_final.groupBy("product_id","year").agg(sum("average_daily_sales").alias("sales")).orderBy("product_id","year")

display(df_final)
